# Manchester Syntax as RDF — User Guide

`starlayergraph` represents a Manchester Syntax document's own internal AST (frames, clauses, class/property expressions, data ranges) as RDF, using the `manch:` ontology - mirroring the sibling `starsparql` package's `salg:` treatment of SPARQL query algebra (see the [SPARQL queries as RDF guide](05c-sparql-query-as-rdf.ipynb) for that design).

See the [Manchester Syntax guide](05f-manchester-syntax.ipynb) for ordinary parsing/serialization - this guide is specifically about treating the *document's own structure* as data you can encode, inspect, edit, and validate, a separate concern from just parsing text into OWL RDF or serializing OWL RDF back to text.

**Fidelity scope**: this matches the parser's own internal AST exactly, the same bar `salg:` itself holds for SPARQL - `onlysome`/`that`/`SuperClassOf:` spelling desugar the same way they already do at parse time (with one exception: `SuperClassOf:`/`SuperPropertyOf:` carry a `manch:reversed` flag, since - unlike other spelling choices - which keyword was written changes the compiled OWL semantics, not just how it reads).

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. This guide's validation step needs `pyshacl` (`pip install -e packages/graph[test]` from a local checkout).
3. Run cells from top to bottom — later steps reuse variables from earlier ones.

## Step 1 — a Manchester Syntax document

Everything below is grounded in this one small ontology fragment, so later steps' effects show up as a real, different compiled meaning — not just triple counts.

In [1]:
from starlayergraph.parsers.manchester_parser import parse_manchester

HEADER = 'Prefix: : <http://example.org/vehicles#>\n'

text = HEADER + """
Class: Vehicle
Class: Car
    SubClassOf: Vehicle

ObjectProperty: hasWheelCount
"""

original_triples = parse_manchester(text)
for t in original_triples:
    print(t)

(rdflib.term.URIRef('http://example.org/vehicles#Vehicle'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#Class'))
(rdflib.term.URIRef('http://example.org/vehicles#Car'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#Class'))
(rdflib.term.URIRef('http://example.org/vehicles#Car'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://example.org/vehicles#Vehicle'))
(rdflib.term.URIRef('http://example.org/vehicles#hasWheelCount'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#ObjectProperty'))


## Step 2 — parse straight to `manch:` RDF

`parse_manchester_ast()` reuses the real parser's own tokenizer/frame/clause/expression grammar completely unchanged - it just captures the AST as RDF via a small set of hooks instead of discarding it after compiling to OWL. `root` is the `manch:Document` node.

In [2]:
from starlayergraph.ontology.to_ast_rdf import parse_manchester_ast

graph, root = parse_manchester_ast(text)
print('encoded triples:', len(graph))
print(graph.serialize(format='turtle'))

encoded triples: 32
@prefix manch: <https://github.com/hidden-graph/starlayergraph/ns/manchester-ast#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

[] a manch:Document ;
    manch:frames ( [ a manch:ClassFrame ;
                manch:clauses () ;
                manch:subject <http://example.org/vehicles#Vehicle> ] [ a manch:ClassFrame ;
                manch:clauses ( [ a manch:ClassSubClassOfClause ;
                            manch:items ( [ a manch:ClauseItem ;
                                        manch:value <http://example.org/vehicles#Vehicle> ] ) ] ) ;
                manch:subject <http://example.org/vehicles#Car> ] [ a manch:ObjectPropertyFrame ;
                manch:clauses () ;
                manch:subject <http://example.org/vehicles#hasWheelCount> ] ) ;
    manch:misc () ;
    manch:prefixes ( [ a manch:PrefixDecl ;
                manch:prefixIri "http://example.org/vehicles#" ;
                manch:prefixLabel ""^^manch:PyStr ] ) .




## Step 3 — edit the document by editing the RDF

Splice a new item into `Car`'s `SubClassOf:` clause's own `manch:items` list, adding `hasWheelCount some xsd:integer` as a second superclass expression - directly as RDF graph surgery, no text-level find/replace anywhere.

In [3]:
from rdflib import RDF, BNode, Namespace
from rdflib.collection import Collection
from rdflib.namespace import XSD
from starlayergraph.ontology.to_ast_rdf import MANCH

EX = Namespace('http://example.org/vehicles#')

sub_class_of_clause = next(graph.subjects(RDF.type, MANCH.ClassSubClassOfClause))
items_list = next(graph.objects(sub_class_of_clause, MANCH.items))

# manch:Some's own onProperty/filler shape (see manchester-ast-ontology.ttl)
some_expr = BNode()
graph.add((some_expr, RDF.type, MANCH.Some))
graph.add((some_expr, MANCH.onProperty, EX.hasWheelCount))
graph.add((some_expr, MANCH.filler, XSD.integer))

new_item = BNode()
graph.add((new_item, RDF.type, MANCH.ClauseItem))
graph.add((new_item, MANCH.value, some_expr))

Collection(graph, items_list).append(new_item)
print('items after edit:', len(list(Collection(graph, items_list))))

items after edit: 2


## Step 4 — decode back into a real `Document` object

`rdf_ast_to_manchester_ast()` decodes the (now-edited) RDF graph into a real `Document` — the same namedtuple-of-namedtuples shape as if it had been built from a `manch:` graph an LLM authored directly, not just one this notebook produced.

In [4]:
from starlayergraph.ontology.to_ast_rdf import rdf_ast_to_manchester_ast

document = rdf_ast_to_manchester_ast(graph, root)
car_frame = next(f for f in document.frames if f.subject == EX.Car)
print(car_frame)

Frame(kind='Class', subject=rdflib.term.URIRef('http://example.org/vehicles#Car'), clauses=[Clause(clause_kind='SubClassOf:', items=[ClauseItem(value=rdflib.term.URIRef('http://example.org/vehicles#Vehicle'), annotations=None), ClauseItem(value=('Some', rdflib.term.URIRef('http://example.org/vehicles#hasWheelCount'), rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')), annotations=None)])])


## Step 5 — re-serialize to Manchester Syntax text

`manchester_ast_to_text()` is a genuinely new serializer (not adapted from the OWL-RDF-based one `g.serialize(format="manchester")` uses) - it's the literal inverse of the parser's own tuple vocabulary, so it can render `Car`'s edited `SubClassOf:` clause with both items on one line, exactly as if that had been typed by hand.

In [5]:
from starlayergraph.ontology.to_ast_rdf import manchester_ast_to_text

edited_text = manchester_ast_to_text(document)
print(edited_text)

Prefix: : <http://example.org/vehicles#>

Class: <http://example.org/vehicles#Vehicle>

Class: <http://example.org/vehicles#Car>
    SubClassOf: <http://example.org/vehicles#Vehicle>, <http://example.org/vehicles#hasWheelCount> some <http://www.w3.org/2001/XMLSchema#integer>

ObjectProperty: <http://example.org/vehicles#hasWheelCount>



## Step 6 — prove the edit was structural

Compile *both* the original and the edited text through the real, unmodified `parse_manchester()` and compare the resulting OWL-semantic triples. The edited version should have new triples this compiled OWL RDF - the new `owl:someValuesFrom` restriction on `Car` - and so should genuinely differ, proving the graph-surgery edit changed real ontology semantics, not just cosmetic layout.

In [6]:
from starlayergraph.compare import isomorphic

edited_triples = parse_manchester(edited_text)

print('original triple count:', len(original_triples))
print('edited triple count:  ', len(edited_triples))
print('isomorphic (should be False - the edit is a real semantic change):', isomorphic(original_triples, edited_triples))

original triple count: 4
edited triple count:   8
isomorphic (should be False - the edit is a real semantic change): False


## Step 7 — load the ontology and shapes graphs

`starlayergraph.ontology.ontology_graph()` and `starlayergraph.ontology.manchester_shapes.shapes_graph()` each return a fresh plain `rdflib.Graph`, not a `StarLayerGraph` - they hold the `manch:` vocabulary/shape definitions themselves, not any document's own data.

In [7]:
from starlayergraph.ontology import ontology_graph
from starlayergraph.ontology import manchester_shapes

ontology = ontology_graph()
print('ontology triples:', len(ontology))

shapes = manchester_shapes.shapes_graph()
print('shapes triples:', len(shapes))

ontology triples: 238
shapes triples: 552


## Step 8 — validate the edited document, with RDFS reasoning over the ontology

`manchester_shapes.validate()` runs the shapes graph against a data graph with `inference="rdfs"` and `ont_graph=ontology_graph()` - real RDFS reasoning before SHACL validation, the same mechanism `starsparql.validate()` uses, and for the same reason: it's what lets a dispatch shape like `manch:ClassExpressionShape` check a single abstract superclass instead of enumerating every concrete tag.

Then deliberately corrupt one triple - retype the spliced-in `manch:Some` restriction as something that isn't a recognized class-expression tag at all - and confirm a real SHACL violation is reported, not a silent pass.

In [8]:
conforms, results_graph, results_text = manchester_shapes.validate(graph)
print('conforms:', conforms)

conforms: True


In [9]:
graph.remove((some_expr, RDF.type, MANCH.Some))

conforms, results_graph, results_text = manchester_shapes.validate(graph)
print('conforms after corruption (should be False):', conforms)
print(results_text)

conforms after corruption (should be False): False
Validation Report
Conforms: False
Results (1):
Constraint Violation in NodeConstraintComponent (http://www.w3.org/ns/shacl#NodeConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:node manch:ClassExpressionItemListShape ; sh:path manch:items ]
	Focus Node: [ manch:items ( [ manch:value <http://example.org/vehicles#Vehicle> ; rdf:type manch:ClauseItem, rdfs:Resource ] [ manch:value [ manch:filler xsd:integer ; manch:onProperty <http://example.org/vehicles#hasWheelCount> ; rdf:type rdfs:Resource ] ; rdf:type manch:ClauseItem, rdfs:Resource ] ) ; rdf:type manch:ClassSubClassOfClause, manch:Clause, rdfs:Resource ]
	Value Node: ( [ manch:value <http://example.org/vehicles#Vehicle> ; rdf:type manch:ClauseItem, rdfs:Resource ] [ manch:value [ manch:filler xsd:integer ; manch:onProperty <http://example.org/vehicles#hasWheelCount> ; rdf:type rdfs:Resource ] ; rdf:type manch:ClauseItem, rdfs:Resource ] )
	Result Path: manch:items
	M

## Further work

- The `manch:` vocabulary and shapes are shipped as real standalone `.ttl` files (`starlayergraph/ontology/manchester-ast-ontology.ttl`/`manchester_shapes.ttl`), usable by any external RDF tool with zero Python dependency.
- See the [SPARQL queries as RDF guide](05c-sparql-query-as-rdf.ipynb) for the sibling design this one mirrors, and the [Manchester Syntax guide](05f-manchester-syntax.ipynb) for ordinary text-to-OWL-RDF parsing and serialization.